# Create EXxx Load Pattern in ETABS
Clones existing `EX` (ASCE 7-05 auto-seismic) with **I changed from 1.25 → 1.0**

**Requirements:** ETABS running with model open · `pip install comtypes`

In [ ]:
# Create EXxx Load Pattern in ETABS
# Clones EX (ASCE 7-05) with I changed 1.25 -> 1.0
# Requirements: ETABS open with model | pip install comtypes

import comtypes.client

# ── 1. CONNECT ──────────────────────────────────────────────────────────────
helper       = comtypes.client.CreateObject('ETABSv1.Helper')
helper       = helper.QueryInterface(comtypes.gen.ETABSv1.cHelper)
etabs_object = helper.GetObject('CSI.ETABS.API.ETABSObject')
model        = etabs_object.SapModel

ver      = model.GetVersion()
filename = model.GetModelFilename(True)
print(f'Connected : ETABS {ver[0]}')
print(f'Model     : {filename}')

# ── 2. SET UNITS & VERIFY EX EXISTS ─────────────────────────────────────────
ret = model.SetPresentUnits(6)  # kN_m
assert ret == 0, f'SetPresentUnits failed: ret={ret}'

t        = model.LoadPatterns.GetNameList()
existing = list(t[1]) if t[0] > 0 else []
print(f'Load patterns: {existing}')
assert 'EX' in existing, "ERROR: source pattern 'EX' not found in model"
if 'EXxx' in existing:
    print('WARNING: EXxx already exists - will be overwritten')

# ── 3. READ ASCE 7-05 TABLE ──────────────────────────────────────────────────
TABLE_NAME = 'Load Pattern Definitions - Auto Seismic - ASCE 7-05'

raw    = model.DatabaseTables.GetTableForDisplayArray(TABLE_NAME, [], 'All', 0, [], 0, [])
fields = [f for f in list(raw[2]) if f is not None]
n_rows = raw[3]
flat   = list(raw[4])
nf     = len(fields)
rows   = [{fields[j]: flat[i*nf+j] for j in range(nf)} for i in range(n_rows)]

ex_rows = [r for r in rows if r.get('Name') == 'EX' and r.get('XDir') is not None]
assert ex_rows, 'ERROR: EX row not found in seismic table'
ex_row = ex_rows[0]

print('\nEX parameters:')
for k, v in ex_row.items():
    if v is not None:
        print(f'  {k:22s}: {v}')

# ── 4. UNLOCK MODEL (clears analysis results) ────────────────────────────────
print(f'\nModel locked: {model.GetModelIsLocked()}')
ret = model.SetModelIsLocked(False)
assert ret == 0, f'Unlock failed: ret={ret}'
print('Model unlocked - analysis results cleared')

# ── 5. ADD EXxx PATTERN SHELL ────────────────────────────────────────────────
# LoadPatterns.Add(Name, Type, SelfWtMultiplier, AddLoadCase)  Type 5 = Seismic
NEW_NAME = 'EXxx'
t2       = model.LoadPatterns.GetNameList()
current  = list(t2[1]) if t2[0] > 0 else []
if NEW_NAME not in current:
    ret = model.LoadPatterns.Add(NEW_NAME, 5, 0.0, True)
    assert ret == 0, f'LoadPatterns.Add failed: ret={ret}'
    print(f"Pattern '{NEW_NAME}' added (Type=5/Seismic, SelfWt=0.0, AddLoadCase=True)")
else:
    print(f"Pattern '{NEW_NAME}' already exists - skipping Add")

# ── 6. BUILD NEW ROW: clone EX, set Name=EXxx, I=1.0 ────────────────────────
new_row         = dict(ex_row)
new_row['Name'] = NEW_NAME
new_row['I']    = '1'   # changed from 1.25 to 1.0

print(f"\nNew '{NEW_NAME}' parameters:")
for k in ['Name','XDir','EccRatio','TopStory','BotStory',
          'PeriodType','CtAndX','SsAndS1From',
          'Ss','S1','TL','SiteClass','Fa','Fv',
          'SDS','SD1','R','Omega','Cd','I']:
    orig = ex_row.get(k)
    new  = new_row.get(k)
    flag = '  <- CHANGED' if new != orig else ''
    print(f'  {k:22s}: {new}{flag}')

# ── 7. RE-READ TABLE POST-ADD, REBUILD FLAT DATA ────────────────────────────
raw2    = model.DatabaseTables.GetTableForDisplayArray(TABLE_NAME, [], 'All', 0, [], 0, [])
fields2 = [f for f in list(raw2[2]) if f is not None]
n2      = raw2[3]
flat2   = list(raw2[4])
nf2     = len(fields2)
rows2   = [{fields2[j]: flat2[i*nf2+j] for j in range(nf2)} for i in range(n2)]

rows_out = [r for r in rows2 if r.get('Name') != NEW_NAME]  # drop old EXxx if exists
rows_out.append(new_row)                                     # append new EXxx

# flatten - comtypes requires tuple (not list) for SAFEARRAY input
flat_out = tuple(r.get(f) for r in rows_out for f in fields2)
print(f'\nRows to write : {len(rows_out)}')

# ── 8. STAGE EDIT via SetTableForEditingArray ────────────────────────────────
ret_stage = model.DatabaseTables.SetTableForEditingArray(
    TABLE_NAME,
    0,
    tuple(fields2),  # field names - must be tuple
    len(rows_out),   # row count
    flat_out         # flat data   - must be tuple
)
print(f'SetTableForEditingArray: {ret_stage[0]}  (expected 1 = staged OK)')
assert ret_stage[0] == 1, f'Staging failed: {ret_stage}'

# ── 9. APPLY EDITS via ApplyEditedTables ─────────────────────────────────────
ret_apply = model.DatabaseTables.ApplyEditedTables(True)
n_fatal   = ret_apply[0]
n_warn    = ret_apply[1]
n_info    = ret_apply[2]
log       = ret_apply[3]
print(f'Fatal errors : {n_fatal}')
print(f'Warnings     : {n_warn}')
print(f'Info msgs    : {n_info}')
for line in log.splitlines():
    if any(w in line.lower() for w in ['record', 'error', 'warning']):
        print(f'  LOG: {line.strip()}')
assert n_fatal == 0, 'ApplyEditedTables had fatal errors - check log above'

# ── 10. VERIFY ───────────────────────────────────────────────────────────────
raw3    = model.DatabaseTables.GetTableForDisplayArray(TABLE_NAME, [], 'All', 0, [], 0, [])
fields3 = [f for f in list(raw3[2]) if f is not None]
n3      = raw3[3]
flat3   = list(raw3[4])
nf3     = len(fields3)
rows3   = [{fields3[j]: flat3[i*nf3+j] for j in range(nf3)} for i in range(n3)]

exxx_row = next((r for r in rows3 if r.get('Name') == NEW_NAME and r.get('I') is not None), None)
assert exxx_row, f"ERROR: '{NEW_NAME}' not found after apply"

print(f"\n'{NEW_NAME}' confirmed in ETABS:")
for k in ['Name','XDir','EccRatio','TopStory','BotStory',
          'PeriodType','CtAndX','SsAndS1From',
          'Ss','S1','TL','SiteClass','Fa','Fv',
          'SDS','SD1','R','Omega','Cd','I']:
    print(f'  {k:22s}: {exxx_row.get(k)}')

t_final      = model.LoadPatterns.GetNameList()
all_patterns = list(t_final[1]) if t_final[0] > 0 else []
print(f'\nAll load patterns : {all_patterns}')
print(f"\nSUCCESS: '{NEW_NAME}' created with I=1.0")
print('Model is unlocked - re-run analysis in ETABS (F5) when ready.')